# M2 Deep Learning Coursework: Normalizing Flows

This notebook is a scaffold for the final coursework implementation.
All major required sections are present as TODO placeholders.


## 0) Setup and Reproducibility


In [21]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt 
import numpy as np
import torch
from IPython.display import Image, Markdown, display


In [22]:
# TODO: keep this seed and deterministic setup in the final implementation.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

# CPU-only coursework assumption
DEVICE = torch.device('cpu')


## 1) Data Loading


In [23]:
# Load CSV files; class labels are retained only for diagnostics (not training).
def load_moons_csv(path: Path):
    arr = np.genfromtxt(path, delimiter=',', names=True, dtype=np.float64)
    x = np.stack([arr['x1'], arr['x2']], axis=1).astype(np.float64)
    cls = arr['class'].astype(np.int64)
    return torch.from_numpy(x).to(DEVICE), torch.from_numpy(cls)
train_x, train_class = load_moons_csv(Path('data/moons_train.csv'))
val_x, val_class = load_moons_csv(Path('data/moons_val.csv'))
test_x, test_class = load_moons_csv(Path('data/moons_test.csv'))
for name, split_x in [('train', train_x), ('val', val_x), ('test', test_x)]:
    if split_x.ndim != 2 or split_x.shape[1] != 2:
        raise ValueError(f'{name} split must have shape [N, 2], got {tuple(split_x.shape)}')
print(f'train/val/test sizes: {len(train_x)}/{len(val_x)}/{len(test_x)}')


train/val/test sizes: 800/100/100


## 1b) Data Exploration

In [ ]:
from scipy.stats import gaussian_kde

# ── Scatter plots coloured by class ──────────────────────────────────────────
splits = {'train': (train_x, train_class), 'val': (val_x, val_class), 'test': (test_x, test_class)}
colors = ['#4C72B0', '#DD8452']

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharex=True, sharey=True)
for ax, (name, (X, Y)) in zip(axes, splits.items()):
    Xn = X.numpy()
    for c, col in zip([0, 1], colors):
        m = (Y == c).numpy()
        ax.scatter(Xn[m, 0], Xn[m, 1], c=col, s=10, alpha=0.7, label=f'class {c}')
    ax.set_title(f'{name}  (N={len(X)})'); ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
    ax.legend(fontsize=8)
fig.suptitle('Dataset splits coloured by class', fontsize=12)
fig.tight_layout()
fig.savefig('figs/Figure_data_scatter.pdf', bbox_inches='tight')
plt.show()

# ── Marginal histograms (train only) ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
Xn = train_x.numpy()
for i, (ax, lbl) in enumerate(zip(axes, ['x₁', 'x₂'])):
    for c, col in zip([0, 1], colors):
        m = (train_class == c).numpy()
        ax.hist(Xn[m, i], bins=30, color=col, alpha=0.6, label=f'class {c}', density=True)
    ax.set_xlabel(lbl); ax.set_ylabel('density'); ax.set_title(f'Marginal: {lbl}'); ax.legend(fontsize=8)
fig.suptitle('Marginal distributions (train)', fontsize=11)
fig.tight_layout()
fig.savefig('figs/Figure_data_marginals.pdf', bbox_inches='tight')
plt.show()

# ── 2D KDE heatmap ───────────────────────────────────────────────────────────
x1_r = np.linspace(Xn[:, 0].min() - 0.3, Xn[:, 0].max() + 0.3, 200)
x2_r = np.linspace(Xn[:, 1].min() - 0.3, Xn[:, 1].max() + 0.3, 200)
xx_kde, yy_kde = np.meshgrid(x1_r, x2_r)
kde = gaussian_kde(Xn.T)
zz_kde = kde(np.vstack([xx_kde.ravel(), yy_kde.ravel()])).reshape(xx_kde.shape)

fig, ax = plt.subplots(figsize=(6, 5))
pcm = ax.pcolormesh(xx_kde, yy_kde, zz_kde, cmap='Blues', shading='auto')
ax.scatter(Xn[:, 0], Xn[:, 1], c='white', s=6, alpha=0.4)
fig.colorbar(pcm, ax=ax, label='KDE density')
ax.set_title('2D KDE of training data'); ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
fig.tight_layout()
fig.savefig('figs/Figure_data_kde.pdf', bbox_inches='tight')
plt.show()

# ── Summary statistics ────────────────────────────────────────────────────────
print(f"{'Split':<8} {'N':>5}  {'x1 mean':>9} {'x1 std':>8} {'x2 mean':>9} {'x2 std':>8}")
print('-' * 55)
for name, (X, _) in splits.items():
    print(f"{name:<8} {len(X):>5}  {X[:,0].mean():>9.4f} {X[:,0].std():>8.4f} {X[:,1].mean():>9.4f} {X[:,1].std():>8.4f}")
print(f"\nTrain class balance: class0={int((train_class==0).sum())}, class1={int((train_class==1).sum())}")

## 2) Model Definition (2D Affine Coupling Flow)


In [24]:
import math
import torch.nn as nn
class AffineCoupling2D(nn.Module):
    """Single 2D affine coupling layer with configurable binary mask."""
    def __init__(self, *, dim: int, hidden: int, mask: torch.Tensor):
        super().__init__()
        if dim != 2:
            raise ValueError('This coursework implementation expects dim=2.')
        if hidden > 128:
            raise ValueError('Coursework constraint violated: hidden must be <= 128.')
        if mask.shape != (dim,):
            raise ValueError(f'mask must have shape ({dim},), got {tuple(mask.shape)}')
        self.dim = dim
        self.hidden = hidden
        self.register_buffer('mask', mask.to(dtype=torch.float64))
        # Required MLP: Linear(D->H) -> ReLU -> Linear(H->2D), then split into s/t.
        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 2 * dim),
        )
    def _st(self, x: torch.Tensor):
        x_masked = x * self.mask
        st = self.net(x_masked)
        s, t = torch.chunk(st, 2, dim=-1)
        s = torch.tanh(s)
        transform_mask = 1.0 - self.mask
        s = s * transform_mask
        t = t * transform_mask
        return s, t
    def forward(self, z: torch.Tensor):
        """Forward map f: base -> data. Returns transformed output and log|det J_f|."""
        s, t = self._st(z)
        transform_mask = 1.0 - self.mask
        x = z * self.mask + transform_mask * (z * torch.exp(s) + t)
        logdet = s.sum(dim=-1)
        return x, logdet
    def inverse(self, x: torch.Tensor):
        """Inverse map f^{-1}: data -> base. Returns output and log|det J_{f^{-1}}|."""
        s, t = self._st(x)
        transform_mask = 1.0 - self.mask
        z = x * self.mask + transform_mask * ((x - t) * torch.exp(-s))
        logdet = -s.sum(dim=-1)
        return z, logdet
class CouplingFlow2D(nn.Module):
    """Stack of affine coupling layers with alternating masks for D=2."""
    def __init__(self, *, n_layers: int, hidden: int, dim: int = 2):
        super().__init__()
        if n_layers < 1 or n_layers > 8:
            raise ValueError('Coursework constraint violated: n_layers must be in [1, 8].')
        if hidden < 1 or hidden > 128:
            raise ValueError('Coursework constraint violated: hidden must be in [1, 128].')
        self.dim = dim
        self.n_layers = n_layers
        self.hidden = hidden
        layers = []
        for i in range(n_layers):
            mask = torch.tensor([1.0, 0.0]) if i % 2 == 0 else torch.tensor([0.0, 1.0])
            layers.append(AffineCoupling2D(dim=dim, hidden=hidden, mask=mask))
        self.layers = nn.ModuleList(layers)
    def forward(self, z: torch.Tensor):
        x = z
        total_logdet = torch.zeros(z.shape[0], dtype=z.dtype, device=z.device)
        for layer in self.layers:
            x, logdet = layer.forward(x)
            total_logdet = total_logdet + logdet
        return x, total_logdet
    def inverse(self, x: torch.Tensor):
        z = x
        total_logdet = torch.zeros(x.shape[0], dtype=x.dtype, device=x.device)
        for layer in reversed(self.layers):
            z, logdet = layer.inverse(z)
            total_logdet = total_logdet + logdet
        return z, total_logdet
    @staticmethod
    def base_log_prob(z: torch.Tensor):
        log_2pi = math.log(2.0 * math.pi)
        return -0.5 * (z.pow(2) + log_2pi).sum(dim=-1)
    def log_prob(self, x: torch.Tensor):
        z, inv_logdet = self.inverse(x)
        return self.base_log_prob(z) + inv_logdet
# Compliant model size: hidden <= 128, n_layers <= 8.
FLOW_CONFIG = {'dim': 2, 'hidden': 64, 'n_layers': 6}
flow = CouplingFlow2D(
    dim=FLOW_CONFIG['dim'],
    hidden=FLOW_CONFIG['hidden'],
    n_layers=FLOW_CONFIG['n_layers'],
).to(DEVICE).double()
flow.eval()


CouplingFlow2D(
  (layers): ModuleList(
    (0-5): 6 x AffineCoupling2D(
      (net): Sequential(
        (0): Linear(in_features=2, out_features=64, bias=True)
        (1): ReLU()
        (2): Linear(in_features=64, out_features=4, bias=True)
      )
    )
  )
)

## 2b) Untrained Model Diagnostics

In [ ]:
# ── Untrained Model Diagnostics ───────────────────────────────────────────────
# Build a dense 2D grid (reused in all subsequent sections)
GRID_N = 200
x1_g = np.linspace(-3.5, 3.5, GRID_N)
x2_g = np.linspace(-2.8, 2.8, GRID_N)
xx_g, yy_g = np.meshgrid(x1_g, x2_g)
grid_pts = torch.tensor(
    np.stack([xx_g.ravel(), yy_g.ravel()], axis=1), dtype=torch.float64
)

flow.eval()
with torch.no_grad():
    log_p_untrained = flow.log_prob(grid_pts).numpy().reshape(GRID_N, GRID_N)
    z_base_pre = torch.randn(1000, 2, dtype=torch.float64)
    x_untrained_samples, _ = flow.forward(z_base_pre)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
im = ax.pcolormesh(xx_g, yy_g, log_p_untrained, cmap='viridis', shading='auto',
                   vmin=np.percentile(log_p_untrained, 5))
fig.colorbar(im, ax=ax, label='log p(x)')
ax.set_title('Untrained: log p(x) heatmap'); ax.set_xlabel('x₁'); ax.set_ylabel('x₂')

ax = axes[1]
Xn = train_x.numpy(); xs_np = x_untrained_samples.numpy()
ax.scatter(Xn[:, 0], Xn[:, 1], c='#4C72B0', s=12, alpha=0.5, label='train data')
ax.scatter(xs_np[:, 0], xs_np[:, 1], c='#DD8452', s=8, alpha=0.4, label='samples (untrained)')
ax.set_xlim(-4, 4); ax.set_ylim(-3.5, 3.5)
ax.set_title('Untrained: samples vs data'); ax.set_xlabel('x₁'); ax.set_ylabel('x₂'); ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig('figs/Figure_untrained.pdf', bbox_inches='tight')
plt.show()

# Baseline NLL (reference for training sections)
with torch.no_grad():
    train_nll_untrained = -flow.log_prob(train_x).mean().item()
    val_nll_untrained   = -flow.log_prob(val_x).mean().item()
print(f'Untrained NLL — train: {train_nll_untrained:.4f}  |  val: {val_nll_untrained:.4f}')

total_params = sum(p.numel() for p in flow.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}  '
      f'(hidden={FLOW_CONFIG["hidden"]}, n_layers={FLOW_CONFIG["n_layers"]})')

## 3) Correctness Checks


In [25]:
# Correctness check 1: invertibility over all training points.
with torch.no_grad():
    z_train, inv_logdet_train = flow.inverse(train_x)
    x_hat, fwd_logdet_train = flow.forward(z_train)
reconstruction_abs = (x_hat - train_x).abs()
invertibility_max_abs_error = float(reconstruction_abs.max().item())
# Correctness check 2: analytic vs finite-difference inverse log-det on first training example.
epsilon = 1e-4
x0 = train_x[:1].clone()
with torch.no_grad():
    _, analytic_inv_logdet = flow.inverse(x0)
analytic_inv_logdet_value = float(analytic_inv_logdet.item())
jacobian_cols = []
for d in range(x0.shape[1]):
    delta = torch.zeros_like(x0)
    delta[0, d] = epsilon
    with torch.no_grad():
        z_plus, _ = flow.inverse(x0 + delta)
        z_minus, _ = flow.inverse(x0 - delta)
    col = ((z_plus - z_minus) / (2.0 * epsilon)).squeeze(0)
    jacobian_cols.append(col)
jacobian = torch.stack(jacobian_cols, dim=1)
finite_diff_logabsdet = float(torch.log(torch.abs(torch.det(jacobian))).item())
logdet_finite_diff_abs_error = float(abs(finite_diff_logabsdet - analytic_inv_logdet_value))
# Diagnostic figure (two panels) required by coursework.
Path('figs').mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
pointwise_recon_err = reconstruction_abs.max(dim=1).values.cpu().numpy()
axes[0].hist(pointwise_recon_err, bins=40, color='tab:blue', alpha=0.8, edgecolor='white')
axes[0].axvline(invertibility_max_abs_error, color='tab:red', linestyle='--', linewidth=2)
axes[0].set_title('Invertibility Check')
axes[0].set_xlabel('Per-point max |x_hat - x|')
axes[0].set_ylabel('Count')
axes[0].text(
    0.98, 0.95,
    f'max error = {invertibility_max_abs_error:.2e}',
    transform=axes[0].transAxes,
    ha='right',
    va='top',
)
axes[1].bar(['analytic', 'finite diff'], [analytic_inv_logdet_value, finite_diff_logabsdet], color=['tab:green', 'tab:orange'])
axes[1].set_title('Inverse Log-Det Check')
axes[1].set_ylabel('log |det J_{f^{-1}}|')
axes[1].text(
    0.5, 0.95,
    f'abs error = {logdet_finite_diff_abs_error:.2e}',
    transform=axes[1].transAxes,
    ha='center',
    va='top',
)
fig.tight_layout()
fig.savefig('figs/Figure1c.pdf', bbox_inches='tight')
plt.close(fig)
# Save required correctness outputs for this step.
results = {
    'correctness': {
        'invertibility_max_abs_error': float(invertibility_max_abs_error),
        'logdet_finite_diff_abs_error': float(logdet_finite_diff_abs_error),
    }
}
with open('results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))
print('Saved figs/Figure1c.pdf')


{
  "correctness": {
    "invertibility_max_abs_error": 9.992007221626409e-16,
    "logdet_finite_diff_abs_error": 1.4130335790341064e-11
  }
}
Saved figs/Figure1c.pdf


## 4) Training on Tiny Subset (First 128 Rows)

In [ ]:
import torch.optim as optim

# ── Tiny-set Training (first 128 rows) ────────────────────────────────────────
TINY_N = 128
tiny_x = train_x[:TINY_N]

TINY_CONFIG = {'hidden': 64, 'n_layers': 6}
flow_tiny = CouplingFlow2D(dim=2, **TINY_CONFIG).to(DEVICE).double()

LR_TINY = 1e-3
N_STEPS_TINY = 2000

optimizer_tiny = optim.Adam(flow_tiny.parameters(), lr=LR_TINY)
tiny_loss_curve = []

flow_tiny.train()
for step in range(N_STEPS_TINY):
    optimizer_tiny.zero_grad()
    nll = -flow_tiny.log_prob(tiny_x).mean()
    nll.backward()
    torch.nn.utils.clip_grad_norm_(flow_tiny.parameters(), 5.0)
    optimizer_tiny.step()
    tiny_loss_curve.append(float(nll.item()))
    if (step + 1) % 500 == 0:
        print(f'  step {step+1:4d}/{N_STEPS_TINY}  NLL={nll.item():.4f}')

flow_tiny.eval()
tinyset_final_nll = tiny_loss_curve[-1]
print(f'\nTiny-set final NLL: {tinyset_final_nll:.4f}')

# ── Figure 2a: loss curve ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(tiny_loss_curve, color='tab:blue', linewidth=1.2)
ax.set_xlabel('Optimiser step')
ax.set_ylabel('NLL (nats)')
ax.set_title('Tiny-set training loss curve')
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('figs/Figure2a.pdf', bbox_inches='tight')
plt.show()

# ── Post-training diagnostics: heatmap, samples, latent space ────────────────
with torch.no_grad():
    log_p_tiny = flow_tiny.log_prob(grid_pts).numpy().reshape(GRID_N, GRID_N)
    x_samp_tiny, _ = flow_tiny.forward(torch.randn(500, 2, dtype=torch.float64))
    z_tiny, _ = flow_tiny.inverse(tiny_x)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

ax = axes[0]
im = ax.pcolormesh(xx_g, yy_g, log_p_tiny, cmap='viridis', shading='auto',
                   vmin=np.percentile(log_p_tiny, 5))
fig.colorbar(im, ax=ax, label='log p(x)')
ax.scatter(tiny_x.numpy()[:, 0], tiny_x.numpy()[:, 1], c='white', s=12, alpha=0.6)
ax.set_title('Tiny-trained: log p(x) heatmap'); ax.set_xlabel('x₁'); ax.set_ylabel('x₂')

ax = axes[1]
ax.scatter(tiny_x.numpy()[:, 0], tiny_x.numpy()[:, 1], c='#4C72B0', s=15, alpha=0.7, label='tiny data')
ax.scatter(x_samp_tiny.numpy()[:, 0], x_samp_tiny.numpy()[:, 1], c='#DD8452', s=8, alpha=0.4, label='samples')
ax.set_xlim(-4, 4); ax.set_ylim(-3.5, 3.5)
ax.set_title('Tiny-trained: samples vs data'); ax.set_xlabel('x₁'); ax.legend(fontsize=9)

ax = axes[2]
z_np = z_tiny.numpy()
ax.scatter(z_np[:, 0], z_np[:, 1], c=train_class[:TINY_N].numpy(), cmap='coolwarm', s=12, alpha=0.7)
theta = np.linspace(0, 2 * np.pi, 200)
for r in [1, 2, 3]:
    ax.plot(r * np.cos(theta), r * np.sin(theta), 'k--', alpha=0.3, linewidth=0.8)
ax.set_title('Tiny-trained: latent space z'); ax.set_xlabel('z₁'); ax.set_ylabel('z₂'); ax.set_aspect('equal')

fig.tight_layout()
fig.savefig('figs/Figure2a_diagnostics.pdf', bbox_inches='tight')
plt.show()

## 5) Full Training

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

# ── Full Training ─────────────────────────────────────────────────────────────
FULL_CONFIG = {'hidden': 64, 'n_layers': 6}
flow_full = CouplingFlow2D(dim=2, **FULL_CONFIG).to(DEVICE).double()

LR_FULL = 1e-3
N_STEPS_FULL = 10_000
BATCH_FULL = 256

optimizer_full = optim.Adam(flow_full.parameters(), lr=LR_FULL)
scheduler = CosineAnnealingLR(optimizer_full, T_max=N_STEPS_FULL, eta_min=1e-5)

full_loss_curve, full_val_curve, lr_curve = [], [], []
rng_full = torch.Generator(); rng_full.manual_seed(SEED)
N_train = len(train_x)

flow_full.train()
for step in range(N_STEPS_FULL):
    idx = torch.randint(0, N_train, (BATCH_FULL,), generator=rng_full)
    xb = train_x[idx]
    optimizer_full.zero_grad()
    nll = -flow_full.log_prob(xb).mean()
    nll.backward()
    torch.nn.utils.clip_grad_norm_(flow_full.parameters(), 5.0)
    optimizer_full.step()
    scheduler.step()
    full_loss_curve.append(float(nll.item()))
    lr_curve.append(scheduler.get_last_lr()[0])
    if (step + 1) % 500 == 0:
        flow_full.eval()
        with torch.no_grad():
            val_nll = -flow_full.log_prob(val_x).mean().item()
        full_val_curve.append((step, val_nll))
        flow_full.train()
        print(f'  step {step+1:5d}/{N_STEPS_FULL}  train NLL={nll.item():.4f}  val NLL={val_nll:.4f}')

flow_full.eval()
Path('checkpoints').mkdir(parents=True, exist_ok=True)
torch.save({'state_dict': flow_full.state_dict(), 'config': FULL_CONFIG, 'seed': SEED},
           'checkpoints/flow_full.pt')
print('Checkpoint saved to checkpoints/flow_full.pt')

# ── Figure 2c: train/val curves + LR schedule ────────────────────────────────
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(full_loss_curve, color='tab:blue', linewidth=0.8, alpha=0.7, label='train NLL (batch)')
val_steps, val_nlls = zip(*full_val_curve)
ax1.plot(val_steps, val_nlls, color='tab:orange', linewidth=2, marker='o', markersize=4, label='val NLL')
ax1.set_xlabel('Optimiser step'); ax1.set_ylabel('NLL (nats)')
ax1.set_title('Full training curves'); ax1.legend(loc='upper right', fontsize=9); ax1.grid(True, alpha=0.3)
ax2 = ax1.twinx()
ax2.plot(lr_curve, color='tab:green', linewidth=0.8, alpha=0.5, linestyle='--')
ax2.set_ylabel('Learning rate', color='tab:green')
ax2.tick_params(axis='y', labelcolor='tab:green')
fig.tight_layout()
fig.savefig('figs/Figure2c.pdf', bbox_inches='tight')
plt.show()

# ── Post-training diagnostics: heatmap, samples, latent space ────────────────
with torch.no_grad():
    log_p_full = flow_full.log_prob(grid_pts).numpy().reshape(GRID_N, GRID_N)
    x_samp_full, _ = flow_full.forward(torch.randn(1000, 2, dtype=torch.float64))
    z_full, _ = flow_full.inverse(train_x)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

ax = axes[0]
im = ax.pcolormesh(xx_g, yy_g, log_p_full, cmap='viridis', shading='auto',
                   vmin=np.percentile(log_p_full, 5))
fig.colorbar(im, ax=ax, label='log p(x)')
ax.scatter(train_x.numpy()[:, 0], train_x.numpy()[:, 1], c='white', s=6, alpha=0.4)
ax.set_title('Full-trained: log p(x) heatmap'); ax.set_xlabel('x₁'); ax.set_ylabel('x₂')

ax = axes[1]
ax.scatter(train_x.numpy()[:, 0], train_x.numpy()[:, 1], c='#4C72B0', s=10, alpha=0.5, label='train data')
ax.scatter(x_samp_full.numpy()[:, 0], x_samp_full.numpy()[:, 1], c='#DD8452', s=6, alpha=0.3, label='samples')
ax.set_xlim(-4, 4); ax.set_ylim(-3.5, 3.5)
ax.set_title('Full-trained: samples vs data'); ax.set_xlabel('x₁'); ax.legend(fontsize=9)

ax = axes[2]
z_np = z_full.numpy()
sc = ax.scatter(z_np[:, 0], z_np[:, 1], c=train_class.numpy(), cmap='coolwarm', s=10, alpha=0.6)
theta = np.linspace(0, 2 * np.pi, 200)
for r in [1, 2, 3]:
    ax.plot(r * np.cos(theta), r * np.sin(theta), 'k--', alpha=0.3, linewidth=0.8)
ax.set_title('Full-trained: latent space z'); ax.set_xlabel('z₁'); ax.set_ylabel('z₂'); ax.set_aspect('equal')
fig.colorbar(sc, ax=ax, label='class')

fig.tight_layout()
fig.savefig('figs/Figure2c_diagnostics.pdf', bbox_inches='tight')
plt.show()

# ── Comparison table ──────────────────────────────────────────────────────────
with torch.no_grad():
    train_nll_full = -flow_full.log_prob(train_x).mean().item()
    val_nll_full   = -flow_full.log_prob(val_x).mean().item()

print(f"\n{'Model':<20} {'Train NLL':>12} {'Val NLL':>12}")
print('-' * 46)
print(f"{'Untrained':<20} {train_nll_untrained:>12.4f} {val_nll_untrained:>12.4f}")
print(f"{'Tiny-trained':<20} {tinyset_final_nll:>12.4f} {'—':>12}")
print(f"{'Full-trained':<20} {train_nll_full:>12.4f} {val_nll_full:>12.4f}")

## 6) Flow Surgery (One-Parameter Family)

In [ ]:
# ── Flow Surgery (One-Parameter Family) ───────────────────────────────────────
ckpt = torch.load('checkpoints/flow_full.pt', map_location=DEVICE, weights_only=False)
flow_trained = CouplingFlow2D(dim=2, **ckpt['config']).to(DEVICE).double()
flow_trained.load_state_dict(ckpt['state_dict'])
flow_trained.eval()

# g_alpha: translate along the principal axis of the training data
X_np = train_x.numpy()
eigvals, eigvecs = np.linalg.eigh(np.cov(X_np.T))
direction = torch.tensor(eigvecs[:, -1], dtype=torch.float64)  # largest eigenvector

def g_alpha_forward(x: torch.Tensor, alpha: float):
    return x + alpha * direction.to(x.device), torch.zeros(x.shape[0], dtype=x.dtype, device=x.device)

def g_alpha_inverse(x: torch.Tensor, alpha: float):
    return x - alpha * direction.to(x.device), torch.zeros(x.shape[0], dtype=x.dtype, device=x.device)

def f_alpha_log_prob(x: torch.Tensor, alpha: float) -> torch.Tensor:
    """log p_{f_alpha}(x) = log p_{f0}(g_alpha^{-1}(x)) + log|det J_{g_alpha^{-1}}|"""
    x_pre, g_logdet = g_alpha_inverse(x, alpha)
    return flow_trained.log_prob(x_pre) + g_logdet

def f_alpha_sample(n: int, alpha: float) -> torch.Tensor:
    with torch.no_grad():
        z = torch.randn(n, 2, dtype=torch.float64)
        x0, _ = flow_trained.forward(z)
        x_shifted, _ = g_alpha_forward(x0, alpha)
    return x_shifted

# ── Figure 3b: five panels for alpha in {-2,-1,0,1,2} ────────────────────────
alphas = [-2, -1, 0, 1, 2]
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharex=True, sharey=True)

for ax, alpha in zip(axes, alphas):
    with torch.no_grad():
        log_p_a = f_alpha_log_prob(grid_pts, alpha).numpy().reshape(GRID_N, GRID_N)
    x_s = f_alpha_sample(500, alpha).numpy()
    ax.pcolormesh(xx_g, yy_g, log_p_a, cmap='viridis', shading='auto',
                  vmin=np.percentile(log_p_a, 5))
    ax.scatter(x_s[:, 0], x_s[:, 1], c='white', s=4, alpha=0.4)
    ax.set_title(f'α = {alpha}'); ax.set_xlabel('x₁')

axes[0].set_ylabel('x₂')
fig.suptitle('Flow surgery: f_α = g_α ∘ f₀  (shift along principal axis)', fontsize=12)
fig.tight_layout()
fig.savefig('figs/Figure3b.pdf', bbox_inches='tight')
plt.show()

## 7) FLOP Counting

In [ ]:
def count_flops(*, dim: int, n_layers: int, hidden: int, batch_size: int) -> int:
    """
    Analytical FLOP count for one log p(x) evaluation (inverse pass through full flow).

    Per coupling layer (inverse pass):
      Linear(dim -> hidden)  : 2 * dim * hidden  (MACs, each = 2 FLOPs)
      ReLU(hidden)           : hidden
      Linear(hidden -> 2*dim): 2 * hidden * (2 * dim)
      tanh(s)  [dim elems]   : dim
      apply transform (sub, exp, mul): 3 * dim
      logdet (sum s)         : dim
    Base log_prob: z^2, log(2pi), sum, scale: 3 * dim
    """
    per_layer = (
        2 * dim * hidden +
        hidden +
        2 * hidden * (2 * dim) +
        dim +
        3 * dim +
        dim
    )
    return int(batch_size * (n_layers * per_layer + 3 * dim))

# Coursework model
flops_b1 = count_flops(dim=2, n_layers=6, hidden=64, batch_size=1)
print(f'FLOPs for log p(x), batch=1 : {flops_b1}')
print(f'FLOPs for log p(x), batch=800: {count_flops(dim=2, n_layers=6, hidden=64, batch_size=800)}')

# Sweep table
print(f"\n{'hidden':>8} {'n_layers':>10} {'FLOPs (B=1)':>14}")
print('-' * 36)
for h in [32, 64, 128]:
    for nl in [2, 4, 6, 8]:
        print(f'{h:>8} {nl:>10} {count_flops(dim=2, n_layers=nl, hidden=h, batch_size=1):>14}')

## 8) Artifact Export

In [ ]:
# ── Artifact Export ───────────────────────────────────────────────────────────
Path('logs').mkdir(parents=True, exist_ok=True)

training_curves = {
    'tiny_loss': tiny_loss_curve,
    'full_loss': full_loss_curve,
    'full_val_loss': [[s, v] for s, v in full_val_curve],
}
with open('logs/training_curves.json', 'w', encoding='utf-8') as f:
    json.dump(training_curves, f)

results = {
    'correctness': {
        'invertibility_max_abs_error': float(invertibility_max_abs_error),
        'logdet_finite_diff_abs_error': float(logdet_finite_diff_abs_error),
    },
    'tinyset_final_nll': float(tinyset_final_nll),
    'full_train_final_nll': float(train_nll_full),
    'full_val_final_nll': float(val_nll_full),
    'flops_batch1': count_flops(dim=2, n_layers=6, hidden=64, batch_size=1),
    'model_config': FULL_CONFIG,
    'seed': SEED,
}
with open('results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

# File checklist
required = [
    'results.json',
    'figs/Figure1c.pdf',
    'figs/Figure2a.pdf',
    'figs/Figure2c.pdf',
    'figs/Figure3b.pdf',
    'checkpoints/flow_full.pt',
    'logs/training_curves.json',
]
print('\n── Output file checklist ──')
for p in required:
    print(f"  {'✓' if Path(p).exists() else '✗ MISSING'}  {p}")